# 03 · Train a proxy model end-to-end

We train a small model **in the notebook** to see every part of the loop, then look at the same
run the way a batch job produces it. The real proxy runs (50M–350M, 1–7B tokens) are batch jobs:

```
gw1b train --gpus 1 --time 12:00:00 --config configs/50m.yaml --set run.name=team2-baseline
```

In [ ]:
# --- GW1B setup cell (same in every notebook) -------------------------------------------------
import os, sys, json, time, pathlib
GW1B_HOME    = os.environ.get("GW1B_HOME", "/SEAS/groups/gw1b/gw1b-f2026")
GW1B_SCRATCH = os.environ.get("GW1B_SCRATCH", "/scratch/gw1b-class")
GW1B_GROUP   = os.environ.get("GW1B_GROUP", "/SEAS/groups/gw1b")
USER_DIR     = os.path.join(GW1B_SCRATCH, "users", os.environ.get("USER", "student"))
os.makedirs(USER_DIR, exist_ok=True)
if GW1B_HOME not in sys.path:
    sys.path.insert(0, GW1B_HOME)        # the shared gw1b package
import jax, jax.numpy as jnp, numpy as np
print("JAX", jax.__version__, "| backend:", jax.default_backend(), "| devices:", jax.devices())
print("your scratch dir:", USER_DIR)


## 1. A configuration
Everything about a run is one YAML file (+ command-line overrides). Let's start from `tiny_debug.yaml` and point it at the mini dataset from notebook 02.

In [ ]:
from gw1b.config import load_config, config_summary
mini = os.path.join(USER_DIR, "data", "mini")
cfg = load_config(os.path.join(GW1B_HOME, "configs", "tiny_debug.yaml"), [
    f"data.train_dir={mini}/train", f"data.val_dir={mini}/val", f"data.tokenizer={USER_DIR}/tok-bpe-4k.model",
    "model.vocab_size=4000", "run.name=nb03-tiny", f"run.out_dir={USER_DIR}/runs",
    "run.total_steps=300", "run.log_every=25", "run.eval_every=100", "run.ckpt_every=150",
    "model.dtype=auto",   # bfloat16 on A100/L40S/GH200, float32 on V100 (no bf16 tensor cores) and CPU
])
print(config_summary(cfg))

## 2. Build the model (sharded over all GPUs of the node) and look at it

In [ ]:
from gw1b.sharding import make_mesh, create_model
from gw1b.model import count_params
mesh = make_mesh()
model = create_model(cfg, mesh)
print(f"{count_params(model)/1e6:.2f}M parameters, dtype = {model.cfg.dtype}, attention impl = {model.attn_impl}")
print("mesh:", mesh)
k = model.blocks[0].mlp.up_proj.kernel[...]
print("one weight matrix:", k.shape, k.dtype, "sharding:", k.sharding.spec)

## 3. Train
`gw1b.train.train(cfg)` is exactly what a batch job runs. It logs to `metrics.jsonl` + TensorBoard, checkpoints with Orbax, and resumes if you call it again.

In [ ]:
from gw1b.train import train
summary = train(cfg)              # ~1-2 minutes on CPU, seconds on a GPU
{k: summary[k] for k in ["step", "tokens", "train_loss", "val_loss", "gpu_hours", "usd_equivalent", "flops"]}

## 4. Plot the loss curve from metrics.jsonl (this is what you put in the paper)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
run_dir = os.path.join(cfg.run.out_dir, cfg.run.name)
m = pd.DataFrame([json.loads(l) for l in open(os.path.join(run_dir, "metrics.jsonl"))])
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
m.dropna(subset=["train/loss"]).plot(x="step", y="train/loss", ax=ax[0], legend=False, title="train loss")
if "val/loss" in m: m.dropna(subset=["val/loss"]).plot(x="tokens_seen", y="val/loss", ax=ax[1], marker="o", legend=False, title="val loss vs tokens")
plt.tight_layout(); plt.show()
print("TensorBoard: run  %tensorboard --logdir", os.path.join(run_dir, "tb"), " (port 6006 is in your tunnel)")

## 5. Generate text from the checkpoint

In [ ]:
from gw1b.evaluate import load_run
from gw1b.generate import generate
from gw1b.tokenizer import Tokenizer
model, cfg2, step = load_run(run_dir)
tok = Tokenizer(cfg2.data.tokenizer)
for t in generate(model, tok, ["the", "we can"], max_new_tokens=20, temperature=0.8, top_k=20, seed=0):
    print(repr(t))

## 6. Try an ablation (Team 2 / Team 3 style)
Change one thing, keep everything else fixed, compare loss at equal tokens **and** equal GPU-hours:
```python
cfg_b = load_config(..., ["model.n_kv_heads=4", "run.name=nb03-mha"])      # GQA -> MHA
cfg_c = load_config(..., ["model.activation=gelu", "model.d_ff=768", "run.name=nb03-gelu"])
cfg_d = load_config(..., ["optim.schedule=wsd", "optim.lr=2e-3", "run.name=nb03-wsd"])
```
As a batch job the same thing is  `gw1b train --config configs/50m.yaml --set model.n_kv_heads=8 --set run.name=mha`.

Next: `04_evaluate_and_export.ipynb`.